# Day 15: Getting Ready for the Real World

Everything we've built works nicely in a notebook. But a notebook is like a test kitchen. A restaurant is a different story: suppliers are late, the gas runs out at 8 PM, a hundred orders come in at once, and every plate has to come out right.

Real apps have the same problems. The OpenAI API sometimes times out. The same question gets asked fifty times a day, and you pay for it fifty times. A bug in one step crashes the whole request. Today is about handling all of that.

We'll cover:

- automatic **retries** when a step fails
- **caching**, so repeated work doesn't cost you twice
- handling errors gracefully with a **fallback**
- the **Functional API**, a way to use LangGraph with plain Python functions
- **testing** your graphs
- **tracing** with LangSmith, so you can see what happened in production
- **deploying** a graph so other programs can use it

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")
llm = ChatOpenAI(model=MODEL, temperature=0)

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. Retries: try again before giving up

When you call someone and the line is busy, you don't give up forever. You wait a bit and try again, and if it's still busy, you wait a little longer before the next try.

That's exactly what a `RetryPolicy` does for a node. If the node raises an error, LangGraph waits and runs it again, waiting a bit longer each time (this is called "backoff").

To see it work, we'll fake an API that fails twice and then succeeds.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy

In [ ]:
attempts = {"count": 0}      # a little counter so we can see how many tries happened

In [ ]:
class WeatherState(TypedDict):
    city: str
    weather: str

In [ ]:
def flaky_weather_api(state: WeatherState) -> dict:
    attempts["count"] += 1
    print(f"  attempt {attempts['count']}")

    if attempts["count"] < 3:
        raise ConnectionError("The weather API timed out")    # fails on tries 1 and 2

    return {"weather": f"{state['city']}: 29 C, clear sky"}

In [ ]:
retry_policy = RetryPolicy(
    max_attempts=4,              # try up to 4 times in total
    initial_interval=0.2,        # wait 0.2 seconds before the first retry
    retry_on=ConnectionError,    # only retry this kind of error
)

In [ ]:
builder = StateGraph(WeatherState)

In [ ]:
builder.add_node("weather", flaky_weather_api, retry_policy=retry_policy)

In [ ]:
builder.add_edge(START, "weather")
builder.add_edge("weather", END)

In [ ]:
weather_app = builder.compile()

### Predict before you run

The fake API fails on attempts 1 and 2. With `max_attempts=4`, does the graph succeed? How many attempts are printed?

<details>
<summary>Click to see the answer</summary>

Yes, it succeeds, and you'll see 3 attempts. It succeeds on the third try, so the fourth is never needed.

If you change `max_attempts` to 2, it gives up after the second failure and the error reaches you.
</details>

In [ ]:
print(weather_app.invoke({"city": "Nashik"}))

A word of advice about `retry_on`. Only retry errors that might go away by themselves: timeouts, rate limits, network blips. Don't retry a `KeyError` or a `TypeError`. Those are bugs in your code, and they'll fail the same way every time, so retrying just wastes time.

`ChatOpenAI` also has its own built-in retries for API errors: `ChatOpenAI(model=..., max_retries=3)`.

## 2. Caching: don't pay twice for the same thing

If a student asks "what is a list in Python?" today, and another student asks the exact same thing tomorrow, a good teacher doesn't prepare a whole new lecture. They reuse the notes.

A `CachePolicy` on a node does that. If the node gets exactly the same input again, LangGraph returns the saved result instead of running the node (and calling the LLM) again. You also need to give `compile()` somewhere to keep the cache.

In [ ]:
import time
from langgraph.types import CachePolicy
from langgraph.cache.memory import InMemoryCache

In [ ]:
class DefineState(TypedDict):
    word: str
    meaning: str

In [ ]:
def define(state: DefineState) -> dict:
    meaning = llm.invoke(f"Define '{state['word']}' in 10 words.").content
    return {"meaning": meaning}

In [ ]:
builder = StateGraph(DefineState)

In [ ]:
builder.add_node("define", define, cache_policy=CachePolicy(ttl=600))    # keep results for 10 minutes

In [ ]:
builder.add_edge(START, "define")
builder.add_edge("define", END)

In [ ]:
dictionary_app = builder.compile(cache=InMemoryCache())

### Predict before you run

We'll ask for "serendipity" twice. Which call will be faster, and will the second one cost anything?

<details>
<summary>Click to see the answer</summary>

The second call should be almost instant and costs nothing, because the input is identical and the result comes from the cache. No API call is made.

Ask for a different word and it calls the LLM again, because the cache key is the node's input.
</details>

In [ ]:
for attempt in (1, 2):
    start = time.time()
    result = dictionary_app.invoke({"word": "serendipity"})
    print(f"Call {attempt}: {time.time() - start:.2f} seconds -> {result['meaning']}")

## 3. Failing gracefully

Retries help with temporary problems. But sometimes things are really broken: the API is down for an hour. In that case you don't want the user to see a scary stack trace. You want a polite message.

The approach is simple: catch the error **inside the node**, write it into the state, and route to a fallback node.

```
  START -> primary --ok--> END
              |
            error
              v
          fallback -> END
```

In [ ]:
from typing import Literal

In [ ]:
class SafeState(TypedDict):
    question: str
    answer: str
    error: str

In [ ]:
def primary(state: SafeState) -> dict:
    try:
        if "crash" in state["question"]:
            raise RuntimeError("The main model is unavailable")    # pretend there's an outage
        answer = llm.invoke(state["question"]).content
        return {"answer": answer, "error": ""}
    except Exception as error:
        return {"error": str(error)}

In [ ]:
def fallback(state: SafeState) -> dict:
    return {"answer": "Sorry, I'm having trouble right now. Please try again in a minute."}

In [ ]:
def ok_or_fallback(state: SafeState) -> Literal["fallback", "__end__"]:
    if state.get("error"):
        return "fallback"
    return END

In [ ]:
builder = StateGraph(SafeState)

In [ ]:
builder.add_node("primary", primary)
builder.add_node("fallback", fallback)

In [ ]:
builder.add_edge(START, "primary")
builder.add_conditional_edges("primary", ok_or_fallback)
builder.add_edge("fallback", END)

In [ ]:
safe_app = builder.compile()

In [ ]:
print(safe_app.invoke({"question": "Say hello in Kannada"}))
print()
print(safe_app.invoke({"question": "please crash"}))

LangChain models also have a shortcut for this: `.with_fallbacks()`. If the first model fails, it tries the next one. A common setup is a cheap model first, with a stronger model as backup.

In [ ]:
reliable_llm = ChatOpenAI(model=MODEL, max_retries=2).with_fallbacks(
    [ChatOpenAI(model="gpt-4.1-mini")]
)

print(reliable_llm.invoke("Reply with just the word OK").content)

## 4. The Functional API: LangGraph without drawing a graph

Not everything needs nodes and edges. Sometimes your logic is just "do this, then if that, do the other thing", and it reads best as normal Python. The **Functional API** lets you write exactly that, and still get LangGraph features like checkpointing, streaming and `interrupt()`.

There are two decorators:

- `@task`: one unit of work, like a node. Calling it starts the work and gives you a "future"; call `.result()` to get the value.
- `@entrypoint`: the main workflow function, like a compiled graph.

In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
@task
def write_outline(topic: str) -> str:
    return llm.invoke(f"A 3-bullet outline for a blog post on {topic}. Very short.").content

In [ ]:
@task
def write_intro(topic: str, outline: str) -> str:
    return llm.invoke(f"Write a 2-sentence intro for a blog post on {topic}, following this outline:\n{outline}").content

In [ ]:
@entrypoint(checkpointer=InMemorySaver())
def blog_workflow(topic: str) -> dict:
    outline = write_outline(topic).result()

    # Plain Python logic, no conditional edges needed
    if len(outline) < 10:
        outline = write_outline(topic + " (please give more detail)").result()

    intro = write_intro(topic, outline).result()
    return {"outline": outline, "intro": intro}

In [ ]:
result = blog_workflow.invoke("monsoon trekking near Pune", {"configurable": {"thread_id": "blog-1"}})

print(result["outline"])
print()
print(result["intro"])

If you start several tasks before calling `.result()` on any of them, they run **in parallel**:

In [ ]:
@task
def fun_fact(topic: str) -> str:
    return llm.invoke(f"One fun fact about {topic}, under 15 words.").content

In [ ]:
@entrypoint()
def many_facts(topics: list[str]) -> list[str]:
    futures = [fun_fact(topic) for topic in topics]      # start them all
    return [future.result() for future in futures]       # then collect them all

In [ ]:
for fact in many_facts.invoke(["tigers", "the Moon", "mangoes"]):
    print("-", fact)

Which one should you use? The Graph API is great when the structure itself matters: agents, branching, multi-agent teams, and you want to *see* the diagram. The Functional API is great for straightforward scripts, or for adding LangGraph to code you already have. Both run on the same engine.

## 5. Testing your graphs

LLM output changes from run to run, so people often think AI apps can't be tested. They can. You just test different things:

1. **The logic**: routers and plain-Python nodes are ordinary functions. Test them like any other function.
2. **The path**: check which nodes ran, using `stream_mode="updates"`.
3. **The shape of LLM output**: don't test the exact words. Test that you got a non-empty string, a valid category, the right type.

In [ ]:
# 1. Test a router directly. No graph, no LLM, no cost.
assert ok_or_fallback({"error": "something broke"}) == "fallback"
assert ok_or_fallback({"error": ""}) == END

In [ ]:
# 2. Test the path the graph took
def nodes_visited(app, graph_input) -> list[str]:
    return [list(step)[0] for step in app.stream(graph_input, stream_mode="updates")]

In [ ]:
assert nodes_visited(safe_app, {"question": "please crash"}) == ["primary", "fallback"]

In [ ]:
# 3. Check the shape of a real LLM answer, not its exact words
result = safe_app.invoke({"question": "Name one planet."})
assert isinstance(result["answer"], str)
assert result["answer"].strip() != ""

print("All tests passed")

In a real project, put these in a file like `tests/test_graph.py` and run `pytest` before every deployment.

## 6. Seeing what happened, with LangSmith

When something goes wrong in production, a user says "the bot gave a weird answer yesterday". How do you find out what happened? Which node ran, what prompt was sent, what came back?

**LangSmith** records every run of your graph: each node, each prompt and response, token counts, cost and time. It's the first thing I set up on any real project. It has a free tier at https://smith.langchain.com.

There's no code change needed. Add these lines to your `.env` and restart the kernel:

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=langgraph-course
```

In [ ]:
if os.getenv("LANGSMITH_TRACING") == "true":
    print("LangSmith tracing is ON")
else:
    print("LangSmith tracing is off (that's fine, it's optional)")

## 7. Deploying a graph

Notebooks are for learning. For an app, the graph lives in a normal `.py` file that creates and exposes the compiled graph. Let's write one. The `%%writefile` line at the top saves the cell's contents to a file instead of running it.

In [ ]:
%%writefile day15_agent.py
"""A small agent, organised the way you'd organise it in a real project."""
import os

from dotenv import load_dotenv, find_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import RetryPolicy

load_dotenv(find_dotenv(usecwd=True))

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0, max_retries=2)


@tool
def word_count(text: str) -> int:
    """Count the words in a piece of text."""
    return len(text.split())


tools = [word_count]
llm_with_tools = llm.bind_tools(tools)


def agent(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent, retry_policy=RetryPolicy(max_attempts=3))
builder.add_node("tools", ToolNode(tools, handle_tool_errors=True))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

# The deployment server adds its own memory, so no checkpointer here
graph = builder.compile()

Now we can import it like any other module and use it.

In [ ]:
from day15_agent import graph

In [ ]:
result = graph.invoke({"messages": [("user", "How many words are in 'deploying graphs is easy'?")]})
print(result["messages"][-1].content)

From here, there are two common ways to serve it.

**Option A: the LangGraph server.** Create a file called `langgraph.json` next to `day15_agent.py`:

```json
{
  "dependencies": ["."],
  "graphs": { "agent": "./day15_agent.py:graph" },
  "env": ".env"
}
```

Then in a terminal:

```bash
pip install -U "langgraph-cli[inmem]"
langgraph dev
```

You get a local API server with threads, memory and streaming already built in, plus **LangGraph Studio**, a visual debugger that opens in your browser. It's really nice for watching an agent work step by step.

**Option B: your own FastAPI app.** If your company already uses FastAPI, the graph slots in easily:

```python
from fastapi import FastAPI
from day15_agent import graph

app = FastAPI()

@app.post("/chat")
async def chat(message: str):
    result = await graph.ainvoke({"messages": [("user", message)]})
    return {"reply": result["messages"][-1].content}
```

Run it with `uvicorn main:app --reload`. For memory, compile the graph with a `PostgresSaver` checkpointer.

## 8. Common mistakes

**Mistake 1: Retrying everything.** `retry_on=Exception` will happily retry a bug ten times. Only retry errors that are temporary.

**Mistake 2: Caching things that change.** A cached answer to "what's the weather now?" will be wrong in an hour. Only cache work where the same input should always give the same output, and set a sensible `ttl`.

**Mistake 3: Swallowing errors silently.** Catching an exception and returning nothing hides problems. Always record the error (in the state, or in your logs) so someone can find out what happened.

**Mistake 4: Testing exact LLM wording.** `assert answer == "The capital is Mumbai."` will fail randomly. Test the type, the category, or that a key fact is present.

**Mistake 5: In-memory storage in production.** `InMemorySaver` and `InMemoryStore` lose everything when the server restarts. Use the Postgres versions for anything real.

## 9. A checklist before you go live

Before any agent goes to real users, I go through this list:

- **Secrets**: API keys in `.env` or a secret manager, never in code.
- **Loops**: a `recursion_limit` and your own step counter.
- **Failures**: `RetryPolicy`, `max_retries`, and a fallback node.
- **Cost**: cache repeated calls, trim or summarize long chats, use small models for easy steps.
- **Memory**: Postgres-backed checkpointer and store, not in-memory ones.
- **Safety**: `interrupt()` before anything risky (payments, emails, deleting data).
- **Quality**: structured output for decisions, evaluator loops for important text.
- **Visibility**: LangSmith tracing, and streaming progress to users.
- **Tests**: routers and tools unit-tested, path tests, and LLM output shape checks.

## Wrapping up

`RetryPolicy` handles temporary failures, and `CachePolicy` with a cache avoids paying twice for the same work. Catch real failures inside nodes and route to a fallback, or use `.with_fallbacks()` on the model. The Functional API (`@entrypoint`, `@task`) gives you LangGraph's features with plain Python. Test the logic, the path and the shape of LLM output. Trace with LangSmith. Deploy by moving the graph into a `.py` file, then serve it with `langgraph dev` or FastAPI.

Tomorrow is the last day. We'll build one complete project, a research assistant, that uses almost everything from the whole course.